# AI-Powered Visual Defect Detection System

## Transfer Learning using ResNet18



In [1]:
import torch
import torchvision

from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
from torchvision.models import resnet18

import matplotlib.pyplot as plt

## Define Image Transformations

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

## Load Training and Validation Data

In [3]:
train_dataset = datasets.ImageFolder(
    "../Dataset/train/images",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    "../Dataset/validation/images",
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("Training Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))

Training Images: 1440
Validation Images: 360


## Load Pretrained ResNet18

In [4]:
model = resnet18(weights="DEFAULT")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\KIIT/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


## Modify Final Layer

In [5]:
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    6
)

print(model.fc)

Linear(in_features=512, out_features=6, bias=True)


In [6]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [9]:
epochs = 5

train_losses = []
train_accuracies = []
for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)

    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)

    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )


Epoch [1/5] Loss: 0.9498 Accuracy: 76.11%
Epoch [2/5] Loss: 0.2957 Accuracy: 96.46%
Epoch [3/5] Loss: 0.1818 Accuracy: 97.36%
Epoch [4/5] Loss: 0.1528 Accuracy: 97.50%
Epoch [5/5] Loss: 0.1128 Accuracy: 97.92%


In [10]:
for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)

    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)

    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

Epoch [1/5] Loss: 0.0998 Accuracy: 98.26%
Epoch [2/5] Loss: 0.0780 Accuracy: 98.82%
Epoch [3/5] Loss: 0.0744 Accuracy: 98.75%
Epoch [4/5] Loss: 0.0713 Accuracy: 98.33%
Epoch [5/5] Loss: 0.0620 Accuracy: 98.96%


In [11]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

validation_accuracy = 100 * correct / total

print(f"Validation Accuracy: {validation_accuracy:.2f}%")

Validation Accuracy: 99.17%


In [12]:
torch.save(
    model.state_dict(),
    "../models/resnet18_final.pth"
)

print("Model Saved Successfully")

Model Saved Successfully
